# Using the `textkit` library crate from a notebook

This notebook is an *outside user* of the library crate `textkit` (`src/lib.rs`).
It can only touch the **public API**, exactly like `examples/demo.rs` and `tests/`.

```text
notebook  ──depends on──▶  textkit (lib crate)
                           ├── parse::lines
                           ├── stats::{TextStats, summary, words::word_count}
                           ├── word_count        (re-export)
                           └── report            (facade over private `render`)
```

**Kernel:** [evcxr](https://github.com/evcxr/evcxr) (`evcxr_jupyter`). Run with the working directory = `notebooks/`.

## 1. Load the crate

`:dep` is an evcxr command (not Rust). It adds a Cargo dependency. The path is relative to this notebook's folder, so `..` is the package root that holds `Cargo.toml`.

In [2]:
:dep textkit = { path = ".." }

## 2. Import paths

Same `use` paths as in `examples/demo.rs`. Names come from the crate's *public* module tree.

In [3]:
use textkit::parse;
use textkit::stats::{self, TextStats};

## 3. `parse::lines`: split text into lines

In [4]:
let text = "Rust organizes code into crates\nand crates into modules\n-- like this!";
let lines = parse::lines(text);
lines

["Rust organizes code into crates", "and crates into modules", "-- like this!"]

## 4. `TextStats`: build with `from_text`, read with getters

The struct is `pub`, but its **fields are private**. A struct literal such as `TextStats { words: 1, .. }` does not compile here. Use the constructor and the getters.

In [5]:
let s = TextStats::from_text(text);
(s.words(), s.lines(), s.longest_word_len())

(11, 3, 9)

`TextStats` derives `Debug`, `Clone`, `PartialEq`, `Eq`, so the notebook can print and compare it.

In [5]:
let a = TextStats::from_text("hello rust");
let b = a.clone();
println!("{a:?}");
a == b

TextStats { words: 2, lines: 1, longest_word_len: 5 }


true

## 5. One function, two paths

`word_count` is re-exported at the crate root with `pub use`. Both paths name the same function. Punctuation-only tokens (`--`) are not words.

In [6]:
let sample = "hello -- world !!";
(
    textkit::word_count(sample),                 // short path (re-export)
    textkit::stats::words::word_count(sample),   // full path
)

(2, 2)

## 6. `stats::summary` and `report`

In [7]:
println!("{}", stats::summary(text));
println!("{}", textkit::report(text));

words: 11, lines: 3
== words: 11, lines: 3 ==


## 7. Per-line table

Same idea as `examples/demo.rs`, with the output shaped as a table.

In [8]:
println!("{:>4}  {:<34} {:>5} {:>7}", "line", "text", "words", "longest");
for (i, line) in parse::lines(text).iter().enumerate() {
    let s = TextStats::from_text(line);
    println!(
        "{:>4}  {:<34} {:>5} {:>7}",
        i + 1,
        format!("{line:?}"),
        s.words(),
        s.longest_word_len()
    );
}

line  text                               words longest
   1  "Rust organizes code into crates"      5       9
   2  "and crates into modules"              4       7
   3  "-- like this!"                        2       5


()

## 8. What the notebook cannot reach

These items are private, so a notebook cell that uses them fails to compile:

| Item | Visibility | Error |
|---|---|---|
| `textkit::render::banner` | module `render` is private | `module render is private` |
| `textkit::stats::words::tokens` | `pub(super)` (only `stats`) | `function tokens is private` |
| `textkit::stats::words::is_word` | no `pub` | `function is_word is private` |
| `s.words` (field) | private field | `field words of struct TextStats is private` |

Use `textkit::report` instead of `render::banner`. It is the public facade.